# 03 — Modélisation, GridSearchCV & tracking MLflow

Ce notebook couvre **Milestone 1 & 4**.

- Setup MLflow
- Pipelines anti data leakage (preprocessing + SMOTE + modèle)
- GridSearchCV sur **score métier**
- Logging des résultats et artefacts (ROC, confusion matrix, modèle)

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # monte d'un niveau (../)
from src.config import PATHS
from src.data import load_parquet
from src.metrics import optimal_threshold_cost, get_proba
from src.models import make_cv, gridsearch_dummy, gridsearch_lgbm_smote, gridsearch_xgb_smote, gridsearch_logreg_smote, gridsearch_rf_smote, run_gridsearch

from src.mlflow_utils import init_mlflow, track_run
import mlflow

## 1) Charger dataset préparé

In [2]:
df = load_parquet(PATHS.data_processed / "train_fe.parquet")

assert "TARGET" in df.columns, "Le dataset doit contenir TARGET (uniquement train)."
X = df.drop(columns=["TARGET"])
y = df["TARGET"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(set(X.dtypes))
print(X.select_dtypes(include='object').columns.tolist())



(79999, 125) (20000, 125)
{dtype('bool'), dtype('float64'), dtype('O'), dtype('int64')}
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


## 2) Init MLflow

In [3]:
from datetime import datetime

experiment_name = "credit_scoring"
init_mlflow()

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment(experiment_name)
# mlflow.enable_system_metrics_logging()
# mlflow.config.set_system_metrics_sampling_interval(1)

notebook_run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print("notebook_run_id: ", notebook_run_id)


notebook_run_id:  20260514_154109


C:\apps\anaconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)


In [4]:
cv = make_cv(2)

In [5]:
from src.models import build_preprocessor

print(X.shape, set(X.dtypes))

pre = build_preprocessor(X)
Xt = pre.fit_transform(X)
print(Xt.shape, Xt.dtype)
print(type(Xt))

(99999, 125) {dtype('bool'), dtype('float64'), dtype('O'), dtype('int64')}
(99999, 256) float64
<class 'numpy.ndarray'>


In [6]:
from sklearn.metrics import roc_auc_score
def get_auc_metric(gs, best_model, X_train, y_train, X_test, y_test):
    # 1. AUC TRAIN
    y_train_proba = get_proba(best_model, X_train)
    auc_train = roc_auc_score(y_train, y_train_proba)
    
    # 2. AUC CROSS-VALIDATION
    auc_cv = gs.cv_results_["mean_test_AUC"][gs.best_index_]
    
    # optionnel : écart-type CV
    auc_cv_std = gs.cv_results_["std_test_AUC"][gs.best_index_]
    
    # 3. AUC VALIDATION / TEST
    y_test_proba = get_proba(best_model, X_test)
    auc_test = roc_auc_score(y_test, y_test_proba)

    return {
        "AUC_train": auc_train,
        "AUC_cv": auc_cv,
        "AUC_cv_std": auc_cv_std,
        "AUC_test": auc_test
    }

## 3) LightGBM

In [ ]:
pipe, param_grid = gridsearch_lgbm_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_lightGBM = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_lightGBM, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

auc_all = get_auc_metric(gs, best_model_lightGBM, X_train, y_train, X_test, y_test)

print(gs.cv_results_.keys())

extra_metrics_lightGBM = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


Fitting 2 folds for each of 8 candidates, totalling 16 fits


### Résultat

In [ ]:

gs.best_params_, gs.best_score_, extra_metrics_lightGBM

### MLFlow tracking

In [ ]:
run_name = "lightgbm"
model_type = "boosting"

track_run(
    run_name= run_name,
    model=best_model_lightGBM,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_lightGBM,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)

### SAVE model to .joblib

#### MLFlow Model Repository: récupération du de la dernière version du modèle

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

def get_model_from_mlflow_model_registry(experiment_name, run_name):
    mlflow.set_tracking_uri("file:./mlruns")
    
    # experiment_name = "credit_scoring"
    wanted_run_name = run_name
    
    client = MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)
    
    if experiment is None:
        raise ValueError(f"Experiment introuvable : {experiment_name}")
    
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.runName = '{wanted_run_name}'",
        order_by=["start_time DESC"],
        max_results=1
    )
    
    if not runs:
        raise ValueError(f"Aucun run trouvé avec le nom : {wanted_run_name}")
    
    run = runs[0]
    run_id = run.info.run_id
    
    print("Run récupéré :", run_id)
    print("Nom du run :", run.data.tags.get("mlflow.runName"))
    print("Date début :", run.info.start_time)
    
    best_model = mlflow.sklearn.load_model(
        f"runs:/{run_id}/{wanted_run_name}_model"
    )
    
    print("Data récupérée dans MLFlow")
    
    threshold = float(run.data.params.get("business_threshold_test", 0.5))
    
    threshold_info = {
        "threshold": threshold,
        "run_id": run_id,
        "run_name": run.data.tags.get("mlflow.runName"),
        "model_type": run.data.tags.get("model_type"),
        "notebook_run_id": run.data.tags.get("notebook_run_id"),
    }
    
    artifact = {
        "model": best_model,
        "threshold": threshold,
        "threshold_info": threshold_info,
        "score_name": "business_cost_opt",
    }

    return artifact


In [ ]:
import joblib

# best_model = best_model_lightGBM #gs.best_estimator_

# artifact = {
#     "model": best_model,
#     "threshold": threshold_info["threshold"],
#     "threshold_info": threshold_info,
#     "score_name": "business_cost_opt",
# }

artifact = get_model_from_mlflow_model_registry(experiment_name, run_name)
print(artifact["model"])
print(artifact["threshold"])
print(artifact["threshold_info"])
print(artifact["score_name"])

# joblib.dump(best_model, "../model/model.joblib")
joblib.dump(artifact, "../model/best_model_lightGBM.joblib")

print("✅ Model saved to model.joblib")
print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)

## 4) Baseline DummyClassifier (référence)

In [ ]:
pipe, param_grid = gridsearch_dummy(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_dummyClassif = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_dummyClassif, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

# CV results artifact (utile pour stabilité & soutenance)
cv_results_df = pd.DataFrame(gs.cv_results_)

import inspect

print(inspect.signature(get_auc_metric))

auc_all = get_auc_metric(gs, best_model_dummyClassif, X_train, y_train, X_test, y_test)

extra_metrics_dummyClassif = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


### Résultat

In [ ]:
gs.best_params_, gs.best_score_, extra_metrics_dummyClassif

### MLFlow tracking

In [ ]:
run_name = "dummy_baseline"
model_type = "dummy"

track_run(
    run_name=run_name,
    model=best_model_dummyClassif,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_dummyClassif,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)

## 5) XGBoost 

In [ ]:
pipe, param_grid = gridsearch_xgb_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_xgBoost = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_xgBoost, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

auc_all = get_auc_metric(gs, best_model_xgBoost, X_train, y_train, X_test, y_test)

extra_metrics_xgBoost = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


### Résultat

In [ ]:
gs.best_params_, gs.best_score_, extra_metrics_xgBoost

### MLFlow tracking

In [ ]:
run_name = "xgboost"
model_type = "boosting"

track_run(
    run_name=run_name,
    model=best_model_xgBoost,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_xgBoost,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)


### SAVE model to .joblib

In [ ]:
import joblib

# best_model = best_model_xgBoost #gs.best_estimator_

# artifact = {
#     "model": best_model,
#     "threshold": threshold_info["threshold"],
#     "threshold_info": threshold_info,
#     "score_name": "business_cost_opt",
# }

artifact = get_model_from_mlflow_model_registry(experiment_name, run_name)
print(run_name)
print(artifact)

# joblib.dump(best_model, "../model/model.joblib")
joblib.dump(artifact, "../model/best_model_xgBoost.joblib")

print("✅ Model saved to model.joblib")
print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)

In [ ]:
# !pip freeze > requirements.txt

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
import imblearn
print(imblearn.__version__)

## 6) Logistic Regression + SMOTE

In [ ]:
pipe, param_grid = gridsearch_logreg_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_logRegression = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_logRegression, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

auc_all = get_auc_metric(gs, best_model_logRegression, X_train, y_train, X_test, y_test)

extra_metrics_logRegression = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


### Résultats

In [ ]:
gs.best_params_, gs.best_score_, extra_metrics_logRegression

### MLFlow tracking

In [ ]:
run_name = "logreg_smote"
model_type = "classic"

track_run(
    run_name=run_name,
    model=best_model_logRegression,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_logRegression,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)

In [ ]:
import mlflow

print("tracking uri =", mlflow.get_tracking_uri())

exp = mlflow.get_experiment_by_name("credit_scoring")
print("credit_scoring experiment =", exp)

runs = mlflow.search_runs()
print("Nb runs =", len(runs))
print(runs[["run_id", "experiment_id", "tags.mlflow.runName"]].tail(20))

## 7) RandomForest + SMOTE

In [ ]:
pipe, param_grid = gridsearch_rf_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_randomForest = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_randomForest, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

auc_all = get_auc_metric(gs, best_model_randomForest, X_train, y_train, X_test, y_test)

extra_metrics_randomForest = {
    # "AUC_test": roc_auc_score(y_test, y_proba),
    "AUC_train": auc_all["AUC_train"],
    "AUC_cv": auc_all["AUC_cv"],
    "AUC_cv_std": auc_all["AUC_cv_std"],
    "AUC_test": auc_all["AUC_test"],
    
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
    
    "fit_time": fit_time,
    "predict_time": predict_time
}


### Résultats

In [ ]:
gs.best_params_, gs.best_score_, extra_metrics_randomForest

### MLFlow tracking

In [ ]:
run_name = "rf_smote"
model_type = "classic"

track_run(
    run_name=run_name,
    model=best_model_randomForest,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_randomForest,
    y_test_proba=y_proba,
    y_test_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
    model_type=model_type,
)

## Analyses des résultats
- Comparer via MLflow UI (runs)